# Connector: docs publishers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_docs_publisher.ipynb)

A `DocsPublisher` ships the audit pack where your reviewers read: a Markdown folder, or a Confluence page. Confluence HTTP is **mocked** with `respx`.

In [1]:
%pip install -q pecca respx pyyaml

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
import httpx, respx, yaml

def configure(docs):
    Path("pecca.yaml").write_text(yaml.safe_dump({"version": 1, "projects": {"default": {"governance": {"extends": "none"}, "integrations": {"docs": docs}}}}))
    context.reset()
train_demo(); print("model trained")

model trained


## Markdown folder

In [4]:
configure({"type": "markdown", "dir": "published"})
location = pecca.audit_pack("route_rfi", out="confluence")      # `confluence` = "send to the configured docs publisher"
print(Path(location).name, "|", len(Path(location).read_text().splitlines()), "lines")

Pecca-audit-default-default-route_rfi-v1.md | 271 lines


## Confluence (mocked)
The publisher looks up the page by title under `parent_page_id`; it creates it the first time and updates (version + 1) afterwards.

In [5]:
configure({"type": "confluence", "url": "https://wiki.bank.example", "space": "AI", "parent_page_id": "123", "email": "pecca@bank.example", "token": "test-token"})
with respx.mock:
    respx.get("https://wiki.bank.example/rest/api/content").mock(side_effect=[
        httpx.Response(200, json={"results": []}),
        httpx.Response(200, json={"results": [{"id": "900", "version": {"number": 1}}]})])
    create = respx.post("https://wiki.bank.example/rest/api/content").mock(return_value=httpx.Response(200, json={"id": "900", "_links": {"webui": "/spaces/AI/pages/900"}}))
    update = respx.put("https://wiki.bank.example/rest/api/content/900").mock(return_value=httpx.Response(200, json={"id": "900", "_links": {"webui": "/spaces/AI/pages/900"}}))
    first = pecca.audit_pack("route_rfi", out="confluence")
    second = pecca.audit_pack("route_rfi", out="confluence")
body = json.loads(create.calls[0].request.content)
print("page:", first)
print("created:", body["title"], "| parent:", body["ancestors"], "| representation:", body["body"]["storage"]["representation"])
print("updated: version", json.loads(update.calls[0].request.content)["version"]["number"], "| same page:", first == second)

page: https://wiki.bank.example/spaces/AI/pages/900
created: Pecca audit — default/default/route_rfi v1 | parent: [{'id': '123'}] | representation: storage
updated: version 2 | same page: True
